# Implement the persistance in LangGraph

In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver



In [7]:
load_dotenv()

True

In [14]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [15]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [16]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [17]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [18]:
# create a graph

graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [20]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'BMW'}, config=config1)

{'topic': 'BMW',
 'joke': 'Why do BMW owners make terrible secret agents?\n\nBecause every time they try to go undercover, they end up revving their engines and drawing too much attention!',
 'explanation': 'This joke plays on a stereotype about BMW owners and their cars. BMWs are often associated with performance and power, which means they tend to have loud engines that can be quite noticeable. The humor comes from the idea that a secret agent, whose job is to operate discreetly and blend into the background, would fail miserably in that role due to the very characteristics of their car.\n\nThe punchline suggests that when these BMW owners try to "go undercover," instead of being stealthy, they attract attention by revving their engines—a behavior often linked to enthusiasm for fast cars. The juxtaposition of the serious and shrouded world of espionage with the flashy and attention-grabbing nature of driving a powerful BMW creates a humorous conflict, making it clear why they\'d stru

In [21]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'BMW', 'joke': 'Why do BMW owners make terrible secret agents?\n\nBecause every time they try to go undercover, they end up revving their engines and drawing too much attention!', 'explanation': 'This joke plays on a stereotype about BMW owners and their cars. BMWs are often associated with performance and power, which means they tend to have loud engines that can be quite noticeable. The humor comes from the idea that a secret agent, whose job is to operate discreetly and blend into the background, would fail miserably in that role due to the very characteristics of their car.\n\nThe punchline suggests that when these BMW owners try to "go undercover," instead of being stealthy, they attract attention by revving their engines—a behavior often linked to enthusiasm for fast cars. The juxtaposition of the serious and shrouded world of espionage with the flashy and attention-grabbing nature of driving a powerful BMW creates a humorous conflict, making it cle

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'BMW', 'joke': 'Why do BMW owners make terrible secret agents?\n\nBecause every time they try to go undercover, they end up revving their engines and drawing too much attention!', 'explanation': 'This joke plays on a stereotype about BMW owners and their cars. BMWs are often associated with performance and power, which means they tend to have loud engines that can be quite noticeable. The humor comes from the idea that a secret agent, whose job is to operate discreetly and blend into the background, would fail miserably in that role due to the very characteristics of their car.\n\nThe punchline suggests that when these BMW owners try to "go undercover," instead of being stealthy, they attract attention by revving their engines—a behavior often linked to enthusiasm for fast cars. The juxtaposition of the serious and shrouded world of espionage with the flashy and attention-grabbing nature of driving a powerful BMW creates a humorous conflict, making it cl